# Entropy Core

Entropy Core draws random bytes from quantum measurement and certifies them: it estimates how much unpredictability the measurements really hold, compresses them to exactly that much, and checks the device with a Bell test in the same job. On IBM hardware the bytes come from the Born rule itself; on the emulator you get the same pipeline as an uncertified baseline. (The API names it Comet Quantum RNG.)

Coin Toss is the simplest engine in this family: one qubit, no certificate.

## Setup

In [1]:
import getpass
import os

import matplotlib.pyplot as plt

from moth_atlas import Atlas

ENGINE = "comet-qrng-staging"  # the engine this notebook runs: Atlas V2 (staging). Production: "comet-qrng-v1"

if not os.environ.get("MOTH_API_KEY"):
    os.environ["MOTH_API_KEY"] = getpass.getpass("Atlas API key")

client = Atlas()  # reads MOTH_API_KEY, and MOTH_API_BASE for which API to call

def find(tree, name):
    """The first value under the key `name` anywhere in a JSON result (an exact key first, else the first key
    containing `name`), or None. The engine nests its numbers in sections; this pulls one out by name."""
    def walk(node, match):
        if isinstance(node, dict):
            for key, value in node.items():
                if match(key):
                    return value
            node = list(node.values())
        if isinstance(node, list):
            for value in node:
                if (found := walk(value, match)) is not None:
                    return found
        return None
    found = walk(tree, lambda key: key == name)
    return found if found is not None else walk(tree, lambda key: name in str(key))


def summary(tree, depth=3, prefix=""):
    """Print a short outline of a JSON result: numbers in full; long strings, lists and big tables cut down."""
    for key, value in tree.items():
        name = f"{prefix}{key}"
        if isinstance(value, dict) and depth > 1 and len(value) <= 12:
            summary(value, depth - 1, name + ".")
        elif isinstance(value, dict):
            print(f"{name}: {{{len(value)} entries}}")
        elif isinstance(value, list):
            print(f"{name}: {len(value)} items, first {value[:3]}" if len(value) > 3 else f"{name}: {value}")
        elif isinstance(value, str) and len(value) > 40:
            print(f"{name}: {value[:32]}… ({len(value)} characters)")
        else:
            print(f"{name}: {value}")

## Try it

32 bytes from 12 qubits × 4096 shots on the emulator, plus 20 dice rolls and 4 floats derived from them. `summary` prints an outline of the result (the bytes, the certificate, the Bell witness), with long values cut short; `job.show()` would print all of it, raw counts included.

In [2]:
job = client.jobs.run(
    engine=ENGINE,
    params={
        "num_qubits": 12,
        "shots": 4096,
        "output_bytes": 32,
        "bell_witness": True,
        "derive": {"integers": {"min": 1, "max": 6, "count": 20}, "floats": 4},
        "mode": "emu",
    },
)
summary(job.result)

comet-qrng-staging job b2b726fa submitted
comet-qrng-staging job b2b726fa completed (5s)
output.bell_witness.S: 2.84375
output.bell_witness.caveat: Fixed measurement settings, no s… (216 characters)
output.bell_witness.classical_bound: 2
output.bell_witness.correlators: 4 items, first [{'E': 0.69921875, 'counts': {'00': 1756, '01': 285, '10': 331, '11': 1724}, 'qubits': [12, 13], 'setting': 'E(a,b)', 'sigma': 0.01117043412526934}, {'E': -0.73046875, 'counts': {'00': 292, '01': 1793, '10': 1751, '11': 260}, 'qubits': [14, 15], 'setting': "E(a,b')", 'sigma': 0.010671033153030935}, {'E': 0.69970703125, 'counts': {'00': 1702, '01': 298, '10': 317, '11': 1779}, 'qubits': [16, 17], 'setting': "E(a',b)", 'sigma': 0.011162967060708578}]
output.bell_witness.enabled: True
output.bell_witness.equivalent_depolarising_p: 0
output.bell_witness.kind: chsh_fidelity_witness
output.bell_witness.p_value_local_realist: 7e-323
output.bell_witness.sigma_S: 0.02197307745211932
output.bell_witness.tsirelson_b

## Parameters

What to expect from each:

- `mode`: `emu` samples a simulator (a classical baseline, uncertified); `qpu` samples IBM hardware (your account needs access).
- `num_qubits` (1 to 256) and `shots` (up to 10,000): raw bits = qubits × shots.
- `output_bytes`: how many conditioned bytes you want. You get at most what the certificate says can be extracted.
- `epsilon_log2` (32 to 128): the extractor's security: output is $2^{-\epsilon}$-close to uniform. Higher is safer and yields fewer bytes.
- `derive`: optional unbiased dice (`integers`: `min`, `max`, `count`) and floats in [0, 1) made from the bytes.
- `bell_witness`: add 8 qubits carrying four CHSH Bell pairs, measured in the same job, as a check on the device.
- `public_seed`: the extractor's seed. The default `toeplitz-v1` is uniform (information-theoretic mode); any other string is expanded with SHAKE-256 and labelled computational.
- `require_assumption_free`: certify only on the tier that assumes nothing about shot independence; on counts it yields almost nothing.
- `include_raw_counts`: return the raw counts, so anyone can recompute the output.
- `prev_pulse_hash`, `pulse_index`: chain this draw onto a previous one, as a randomness beacon.

In [3]:
# client.engines.get(ENGINE)  # its description, params, input and output files, from the API

## How it works

**Measure.** Each of `num_qubits` qubits is put into an equal superposition and measured, `shots` times. On hardware, the Born rule decides each bit at the moment of measurement: there is no seed underneath.

**Certify.** Real devices aren't perfect coins. The engine estimates the min-entropy of the raw bits (following NIST SP 800-90B), a lower bound on how unpredictable they are, which is the number of truly random bits they contain.

**Extract.** A Toeplitz-hashing extractor compresses the raw bits to that many output bits, $2^{-\epsilon}$-close to uniform. The output is shorter than the input; the certificate says by how much.

**Witness.** Four Bell pairs measured alongside give a CHSH value $S$. Any classical (local hidden-variable) explanation keeps $S \le 2$; entangled qubits reach up to $2\sqrt{2} \approx 2.83$. A high $S$ is evidence the device was behaving quantumly during the run.

On `emu`, every step runs the same way, but the bits come from a classical pseudo-random generator: useful for testing, not certified randomness.

## Experiments

Each experiment changes one parameter and keeps the others fixed, and runs one job per value.

### Shots and extractable bytes

More raw bits support more certified output. Ask for more than can be extracted and you get what the certificate allows.

In [4]:
for shots in (1024, 4096, 10000):
    print(f"shots={shots}")
    r = client.jobs.run(engine=ENGINE, params={"num_qubits": 12, "shots": shots, "output_bytes": 100000}, quiet=True).result
    print(f"  raw bits {12 * shots}, min-entropy {find(r, 'min_entropy')}, extractable {find(r, 'extractable')}")

shots=1024
  raw bits 12288, min-entropy None, extractable None
shots=4096
  raw bits 49152, min-entropy None, extractable None
shots=10000
  raw bits 120000, min-entropy None, extractable None


### Dice

600 dice rolls derived from one draw. Fair dice land close to 100 per face.

In [5]:
r = client.jobs.run(engine=ENGINE, params={"shots": 10000, "derive": {"integers": {"min": 1, "max": 6, "count": 600}}}).result
rolls = find(r, "integers")
if isinstance(rolls, list):
    plt.figure(figsize=(4, 2.5))
    plt.hist(rolls, bins=[x - 0.5 for x in range(1, 8)], rwidth=0.8)
    plt.axhline(100, color="k", ls="--", lw=1)
    plt.xlabel("face")
    plt.ylabel("rolls")
    plt.show()
else:
    print(rolls)

comet-qrng-staging job 8055ce68 submitted
comet-qrng-staging job 8055ce68 completed (5s)
None


### Bell witness

The CHSH value from the four Bell pairs: above 2 is beyond any classical explanation.

In [6]:
r = client.jobs.run(engine=ENGINE, params={"bell_witness": True, "shots": 4096}, quiet=True).result
print("CHSH:", find(r, "chsh"))

CHSH: None


## Questions

**Is emulator output random enough to use?**

It's high-quality pseudo-randomness, but it isn't certified: a simulator is deterministic underneath. Use `mode: "qpu"` for randomness with a quantum origin.

**Why are there fewer output bytes than raw bits?**

Raw bits from a real device are slightly biased and correlated. The extractor keeps only as many bits as the min-entropy certifies, so the output can be trusted as uniform.

**Can someone else check my output?**

Yes: with `include_raw_counts`, the result carries everything needed to rerun the extractor and get the same bytes.

**What's a beacon pulse?**

A public, timestamped random value that others can rely on. `prev_pulse_hash` and `pulse_index` chain each draw onto the last, so the sequence can't be rewritten later.